In [2]:
%load_ext autoreload
%autoreload 2

# Import

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Union
from glob import glob

import ipywidgets as widgets
from IPython.display import display

import sys
PHASE1_SRC = Path("../Phase1/src/")
if str(PHASE1_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE1_SRC))

from space import Space, GFLOWNET_ENV
from reward import reward_peak, reward_latent
from machinelearning import train_single_model, predict_with_model, preprocess, build_models
from VEM import oracle_fn, TARGET_NAMES
from plotting import *

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from umap import UMAP
from sklearn.decomposition import PCA


# Loading

In [4]:
run = "grid_results_mix_and_match"

In [5]:
grid = pd.read_csv(f"{run}/grid_summary.csv")
grid = grid.rename(columns={"sampling_strategy": "schedule"})
# convert schedule from string to dict
grid["schedule"] = grid["schedule"].apply(eval)

REWARD_FILTER = "reward_peak_new" # "reward_latent" | "reward_peak" | "reward_peak_new"
grid = grid[grid["reward_fn_name"] == REWARD_FILTER]

In [6]:
from glob import glob
paths = sorted(glob(f"{run}/*/al_metrics.csv"))
paths = [p for p in paths if Path(p).parent.name in grid["run"].values] # filters per reward as wreitten above
if not paths:
    raise ValueError("No files found at al_results/*/al_metrics.csv")
df = pd.concat([load(p).assign(run=Path(p).parent.name) for p in paths], ignore_index=True)

In [7]:
df

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,best_reward_this_iter,mean_reward_this_iter,best_predicted_reward,elapsed_sec,run
0,1,16:05:47,70,70,10,0.507243,0.187621,0.371863,0.103033,random,0.433629,0.258543,0.359967,1.3,run_0
1,2,16:05:48,80,80,10,0.507243,0.200188,0.366879,0.087255,random,0.487089,0.288154,0.374669,1.1,run_0
2,3,16:05:48,90,90,10,0.507243,0.215769,0.552121,0.086983,random,0.477274,0.340416,0.394493,0.1,run_0
3,4,16:05:49,100,100,10,0.507243,0.216017,0.365110,0.079793,random,0.353359,0.218256,0.414628,0.1,run_0
4,5,16:05:49,110,110,10,0.507243,0.214851,0.476689,0.076245,random,0.392686,0.203186,0.404573,0.1,run_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52555,56,22:39:43,640,640,5,0.875610,0.243802,0.663225,0.065368,grid,0.450605,0.383469,0.462180,0.2,run_99
52556,57,22:39:43,645,645,5,0.875610,0.244895,0.608653,0.070783,grid,0.446747,0.384798,0.457610,0.2,run_99
52557,58,22:39:43,650,650,5,0.875610,0.244997,0.657826,0.065676,grid,0.355705,0.258158,0.418384,0.2,run_99
52558,59,22:39:44,655,655,5,0.875610,0.245575,0.617050,0.069491,grid,0.418515,0.320638,0.432256,0.2,run_99


In [8]:
df[df["sampling_strategy"]=="gflownet"].groupby("run").count().count().iloc[0]

576

In [9]:
best_runs(df)

,run,final_mean_reward,final_best_reward,final_proxy_r2,final_proxy_mae,total_oracle_evals,sampling_strategy
0,run_86,0.364714,0.897222,0.212117,0.082534,660,genetic
1,run_81,0.241540,0.892366,0.556692,0.072986,660,gflownet
2,run_45,0.255256,0.888781,0.608141,0.071842,660,gp
3,run_98,0.275923,0.888262,0.311880,0.074929,660,grid
4,run_110,0.278642,0.888262,0.383903,0.072573,660,lhs
5,run_2,0.277140,0.888262,0.327806,0.073047,660,random


## Visualization

In [10]:
# filter out elapsed_sec and sampling_strategy fom grid before merging 
metrics = grid.merge(
    df[[c for c in df.columns if c != "elapsed_sec"]],
    on="run",
    how="left",
)

In [11]:
def _schedule_str(sched: dict) -> str:
    if isinstance(sched, str):
        sched = eval(sched)
    return "→".join(f"{k}:{v}" for k, v in sched.items())

def _n_phases(sched: dict) -> int:
    if isinstance(sched, str):
        sched = eval(sched)
    return len(sched)

metrics["schedule_str"] = metrics["schedule"].map(_schedule_str)
metrics["n_phases"] = metrics["schedule"].map(_n_phases)
metrics["is_pure"] = metrics["n_phases"] == 1

In [12]:
pure = metrics[metrics["is_pure"]]
mixed = metrics[~metrics["is_pure"]]

In [13]:
# run_summary = metrics.groupby(["run", "schedule_str"])["best_reward_so_far"].max().reset_index()
# order = run_summary.groupby("schedule_str")["best_reward_so_far"].median().sort_values().index
# sns.boxplot(data=run_summary, x="schedule_str", y="best_reward_so_far", order=order)
# plt.xticks(rotation=90)

In [14]:
# run_df = metrics[metrics["run"] == "run_378"].sort_values("iteration")
# run_df["phase_change"] = run_df["sampling_strategy"].ne(run_df["sampling_strategy"].shift())
# for strat, seg in run_df.groupby((run_df["sampling_strategy"] != run_df["sampling_strategy"].shift()).cumsum()):
#     plt.plot(seg["iteration"], seg["mean_reward_so_far"], label=seg["sampling_strategy"].iloc[0])

In [15]:
select_best_runs(mixed, metric="best_reward", groupby="schedule_str")

,run,status,elapsed_sec,best_reward,mean_reward,total_oracle_budget,schedule,acquisition,reward_fn_name,seed,...,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,best_reward_this_iter,mean_reward_this_iter,best_predicted_reward,schedule_str,n_phases,is_pure
22680,run_378,ok,335.7,0.897176,0.327892,660,"{'gflownet': 30, 'genetic': 30}",diverse_top_k,reward_peak_new,123,...,0.185364,0.371863,0.103033,gflownet,0.446502,0.242743,0.398374,gflownet:30→genetic:30,2,False
22681,run_378,ok,335.7,0.897176,0.327892,660,"{'gflownet': 30, 'genetic': 30}",diverse_top_k,reward_peak_new,123,...,0.202926,0.351089,0.085131,gflownet,0.422802,0.325861,0.383907,gflownet:30→genetic:30,2,False
22682,run_378,ok,335.7,0.897176,0.327892,660,"{'gflownet': 30, 'genetic': 30}",diverse_top_k,reward_peak_new,123,...,0.214599,0.516456,0.083057,gflownet,0.486879,0.307982,0.400350,gflownet:30→genetic:30,2,False
22683,run_378,ok,335.7,0.897176,0.327892,660,"{'gflownet': 30, 'genetic': 30}",diverse_top_k,reward_peak_new,123,...,0.218399,0.331562,0.072542,gflownet,0.360742,0.252595,0.421968,gflownet:30→genetic:30,2,False
22684,run_378,ok,335.7,0.897176,0.327892,660,"{'gflownet': 30, 'genetic': 30}",diverse_top_k,reward_peak_new,123,...,0.226157,0.364402,0.066996,gflownet,0.473689,0.303738,0.397449,gflownet:30→genetic:30,2,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6655,run_110,ok,9.6,0.888262,0.278642,660,"{'random': 30, 'lhs': 30}",top_k,reward_peak_new,456,...,0.277906,0.335546,0.073777,lhs,0.499987,0.332000,0.388451,random:30→lhs:30,2,False
6656,run_110,ok,9.6,0.888262,0.278642,660,"{'random': 30, 'lhs': 30}",top_k,reward_peak_new,456,...,0.278511,0.412309,0.071268,lhs,0.441949,0.316031,0.405074,random:30→lhs:30,2,False
6657,run_110,ok,9.6,0.888262,0.278642,660,"{'random': 30, 'lhs': 30}",top_k,reward_peak_new,456,...,0.278793,0.402209,0.074333,lhs,0.431125,0.296557,0.424317,random:30→lhs:30,2,False
6658,run_110,ok,9.6,0.888262,0.278642,660,"{'random': 30, 'lhs': 30}",top_k,reward_peak_new,456,...,0.278522,0.376761,0.076737,lhs,0.386999,0.261223,0.364078,random:30→lhs:30,2,False


In [16]:
print(metrics[metrics["schedule"] == {'random': 30, 'lhs': 30}]["run"].nunique())
print(metrics[metrics["schedule"] == {'random': 30, 'gflownet': 30}]["run"].nunique())

12
36


There are 3 seeds x 2 budgets x 2 acquisitions (x 3 gfn_budgets)

In [17]:
select_best_runs(pure, metric="best_reward")[["run", "sampling_strategy"]].drop_duplicates()

,run,sampling_strategy
5160,run_86,genetic
4860,run_81,gflownet
2700,run_45,gp
900,run_15,grid
1920,run_32,lhs
120,run_2,random


In [18]:
select_best_runs(mixed, metric="best_reward", groupby="schedule_str")[["run", "schedule_str"]].drop_duplicates()


,run,schedule_str
22680,run_378,gflownet:30→genetic:30
50400,run_840,gp:20→gflownet:20→genetic:20
21000,run_350,gp:30→genetic:30
20340,run_339,gp:30→gflownet:30
43320,run_722,grid:20→gflownet:20→genetic:20
42480,run_708,grid:20→gp:20→genetic:20
40680,run_678,grid:20→gp:20→gflownet:20
39600,run_660,grid:20→lhs:20→genetic:20
37980,run_633,grid:20→lhs:20→gflownet:20
37080,run_618,grid:20→lhs:20→gp:20


In [19]:
plot_reward_curves(select_best_runs(pure, metric="mean_reward"), metric="mean_reward_this_iter", title="Mean reward per iteration", ceiling=True)

In [20]:
agg, symbols = aggregate_seeds_for_scatter(pure)

In [21]:
agg.sample()

,sampling_strategy,acquisition,n_candidates_per_iter,n_init,n_iterations,init_method,gfn_batch_size,gfn_n_train_steps,schedule_str,best_reward,best_reward_std,mean_reward,mean_reward_std,n,run,marker_symbol
8,gflownet,diverse_top_k,10,60,60,random,50,200,gflownet:60,0.748521,0.172621,0.263094,0.022562,180,60 + 10 * 60 | acquisition: diverse_top_k | in...,1


In [22]:
plot_scatter(agg, symbols)

In [23]:
agg, symbols = aggregate_seeds_for_scatter(mixed)
fig2 = plot_scatter(agg, symbols)
fig2.show()

In [24]:
out1 = widgets.Output(layout={'border': '1px solid black', 'width': '50%'})
out2 = widgets.Output(layout={'border': '1px solid black', 'width': '50%'})

with out1:
    agg, symbols = aggregate_seeds_for_scatter(pure)
    fig1 = plot_scatter(agg, symbols)
    fig1.show()
    
with out2:
    agg, symbols = aggregate_seeds_for_scatter(mixed)
    fig2 = plot_scatter(agg, symbols)
    fig2.show()

display(widgets.HBox([out1, out2]))

In [25]:
plot_seed_variance(metrics, metric="mean_reward_this_iter")

In [26]:
plot_oracle_efficiency(metrics, metric="mean_reward_this_iter")

In [27]:
plot_proxy_curves(metrics)

In [28]:
plot_final_boxplot(metrics, metric="best_reward", groupby="schedule_str", color_by="n_phases")

In [29]:
plot_reward_heatmap(metrics, metric="mean_reward", groupby="schedule_str")

In [30]:
plot_budget_tradeoff(metrics, metric="mean_reward", groupby="schedule_str")

In [31]:
embedding, preprocessor, reducer = compute_global_embedding(
    method="umap"
)

/u/porchetv/Desktop/PhD/plasma/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## pure strategies only

In [32]:
sampled_dict_pure = load_best_per_schedule(pure, results_dir=run, preprocessor=preprocessor, reducer=reducer,
                                            groupby="sampling_strategy")  # old behavior, unchanged

In [33]:
plot_sampling_grid(sampled_dict_pure, cols=2)

In [35]:
plot_exploration_dashboard(sampled_dict_pure, preprocessor)

TypeError: plot_exploration_dashboard() missing 1 required positional argument: 'schedules'

In [ ]:
plot_peak_coverage(sampled_dict_pure, "inverse", reward="new")

In [ ]:
fig, table =plot_peak_coverage_heatmap(sampled_dict_pure, reward="new", metric="fraction")
fig.update_layout(height=600, width=800)
fig.show()

## All schedules

In [36]:
top = select_top_runs(metrics, n=41, metric="best_reward", groupby="schedule_str", dedupe=True)

sampled_dict_best = load_best_per_schedule(top, results_dir=run, preprocessor=preprocessor, reducer=reducer,
                                            groupby="schedule_str")
sampled_dict_all = load_pooled_per_schedule(top, results_dir=run, preprocessor=preprocessor, reducer=reducer,
                                             groupby="schedule_str")

In [37]:
schedule_lookup = metrics.drop_duplicates("schedule_str").set_index("schedule_str")["schedule"]

schedules = {name: schedule_lookup[name] for name in sampled_dict_all.keys()}

In [47]:
fig = plot_exploration_dashboard(sampled_dict_all, preprocessor, schedules)
fig.update_layout(width=1000)
fig.show()

In [ ]:
t = phase_transition_table(sampled_dict_all, preprocessor, schedules)
t[t["phase"] == "genetic"].sort_values("reward_mean_delta", ascending=False)

,schedule,phase,phase_start,phase_end,reward_mean,novelty_mean,batch_diversity_mean,coverage_gap_end,reward_mean_delta,novelty_mean_delta,batch_diversity_mean_delta
66,random:20→grid:20→genetic:20,genetic,41,60,0.392552,0.555760,1.397468,0.476900,0.115554,-1.075462,-1.694491
31,grid:30→genetic:30,genetic,31,60,0.389013,0.506563,0.949122,0.499674,0.109575,-1.319530,-2.080468
57,random:20→gflownet:20→genetic:20,genetic,41,60,0.390751,0.464496,0.909101,0.496824,0.105307,-1.128956,-2.825755
78,random:20→lhs:20→genetic:20,genetic,41,60,0.398904,0.437405,0.975151,0.500180,0.103079,-1.189970,-2.850701
49,lhs:30→genetic:30,genetic,31,60,0.383513,0.503995,1.254422,0.514063,0.099624,-1.300703,-2.513047
2,gflownet:30→genetic:30,genetic,31,60,0.391310,0.567040,1.779465,0.509222,0.096418,-1.238327,-1.907212
14,grid:20→gflownet:20→genetic:20,genetic,41,60,0.385340,0.646283,1.440197,0.498388,0.095170,-0.952618,-2.381137
23,grid:20→lhs:20→genetic:20,genetic,41,60,0.382550,0.539533,1.154747,0.485460,0.088507,-1.079139,-2.664016
8,gp:30→genetic:30,genetic,31,60,0.399668,0.404536,1.166508,0.549218,0.088022,-1.246090,-2.232628
6,gp:20→gflownet:20→genetic:20,genetic,41,60,0.392698,0.611221,1.648202,0.525267,0.083595,-0.922274,-1.966074


In [ ]:
plot_exploration_heatmap(sampled_dict_all, preprocessor, schedules)

In [ ]:
# plot_peak_coverage(sampled_dict_all, 'inverse', reward="new")

In [ ]:
fig, table = plot_peak_coverage_heatmap(sampled_dict_all, reward="new", metric="fraction")
fig.update_layout(height=600, width=800)
fig.show()

In [ ]:
df = metrics[metrics["iteration"] > 0]


# plot
fig = go.Figure()
for i, val in enumerate(sorted(df["gfn_loss"].dropna().unique())):
    agg = df[df["gfn_loss"] == val].groupby("iteration")["mean_reward_this_iter"].agg(["mean","std"]).reset_index()
    c = f"hsl({i * 60}, 70%, 50%)"
    fig.add_trace(go.Scatter(x=agg["iteration"], y=agg["mean"], name=str(val),
                                line=dict(color=c, width=2), mode="lines+markers"))
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["iteration"], agg["iteration"][::-1]]),
        y=pd.concat([agg["mean"]+agg["std"], (agg["mean"]-agg["std"])[::-1]]),
        fill="toself", fillcolor=c, opacity=0.15, line=dict(width=0),
        showlegend=False, hoverinfo="skip"))
fig.update_layout(title=f"{'mean_reward_this_iter'} by {'gfn_loss'}", xaxis_title="iteration",
                    plot_bgcolor="white", hovermode="x unified")